In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]", "executable": "/home/miakhlov/course-production-verified-20261118/scripts/environment/.venv-chem/bin/python", "platform": "Linux-6.18.58.1-microsoft-standard-WSL2-x86_64-with-glibc2.39", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0", "httpcore": "1.0.9", "httpx": "0.28.1", "idna": 

# Разбор A. Реальные температурные данные, растворитель и твёрдая фаза

Table S1 [Wang et al.](https://doi.org/10.3390/cryst12070917) содержит 60 строк: 12 растворителей/смесей × 5 температур 293.15–333.15 K. Методы: 12 h перемешивание и ещё 12 h выдержка; среднее трёх опытов, без raw repeats/SD в таблице. Reported equilibrium forms различны: I, гидрат II, сольват III. Смешивать их как одну твёрдую фазу нельзя. **100δ — отклонение авторской модели, не ошибка измерения.**

Фитируем ln x=a+b·(1000/T−mean(1000/T)). Apparent slope quantity ΔH_app=−R·b в kJ/mol, если R выражена в J mol⁻¹ K⁻¹. Это условный параметр выбранной концентрационной основы; thermodynamic enthalpy требует activities, стабильности фаз и согласованных исходных данных.

In [2]:
from pathlib import Path
import sys,json,csv,math
import numpy as np
from scipy import stats
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=workspace('M04-L02','case');S_DATA=DATA/'istradefylline_solubility'
def read_csv(name):
    with (S_DATA/name).open(encoding='utf-8',newline='') as f:return list(csv.DictReader(f))
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=150,bbox_inches='tight');display(fig);plt.close(fig)


In [3]:
rows=read_csv('istradefylline_solubility_table_S1.csv')
solvents=list(dict.fromkeys(r['solvent'] for r in rows));fits=[]
for solvent in solvents:
    group=[r for r in rows if r['solvent']==solvent]
    T=np.array([float(r['temperature_K']) for r in group]);x=np.array([float(r['x_as_reported']) for r in group])
    z=1000/T;X=np.column_stack([np.ones(len(T)),z-z.mean()]);y=np.log(x)
    beta=np.linalg.lstsq(X,y,rcond=None)[0];residual=y-X@beta;df=len(y)-2
    inverse=np.linalg.inv(X.T@X);mse=float(residual@residual/df)
    se=np.sqrt(np.diag(mse*inverse));critical=stats.t.ppf(.975,df)
    h=np.einsum('ij,jk,ik->i',X,inverse,X)
    fits.append({'solvent':solvent,'form':group[0]['equilibrium_form_as_reported'],
                 'apparent_slope_kJ_mol':float(-8.31446261815324*beta[1]),
                 'conditional_half_CI95_kJ_mol':float(8.31446261815324*critical*se[1]),
                 'leave_one_temperature_out_RMSE_ln_x':float(np.sqrt(np.mean((residual/(1-h))**2)))})
display(fits)


[{'solvent': 'methanol',
  'form': 'II',
  'apparent_slope_kJ_mol': 66.2327912241552,
  'conditional_half_CI95_kJ_mol': 1.5217790697836884,
  'leave_one_temperature_out_RMSE_ln_x': 0.023843815463749488},
 {'solvent': 'ethanol',
  'form': 'I',
  'apparent_slope_kJ_mol': 60.05851331049946,
  'conditional_half_CI95_kJ_mol': 8.416927517649071,
  'leave_one_temperature_out_RMSE_ln_x': 0.15450119792997932},
 {'solvent': 'ethyl acetate',
  'form': 'I',
  'apparent_slope_kJ_mol': 28.02305247391376,
  'conditional_half_CI95_kJ_mol': 4.82018009084367,
  'leave_one_temperature_out_RMSE_ln_x': 0.0855519649862451},
 {'solvent': 'n-propanol',
  'form': 'I',
  'apparent_slope_kJ_mol': 47.31564713090255,
  'conditional_half_CI95_kJ_mol': 11.294025485656745,
  'leave_one_temperature_out_RMSE_ln_x': 0.15315955566847392},
 {'solvent': 'isopropanol',
  'form': 'I',
  'apparent_slope_kJ_mol': 35.67790712981914,
  'conditional_half_CI95_kJ_mol': 6.223916344152023,
  'leave_one_temperature_out_RMSE_ln_x': 0.

Проверка source consistency важнее выбора «лучшего» растворителя. Отдельно пересчитаем 100|x_calc−x|/x; значительные расхождения с 100δ требуют review. Не используем δ как вес. Если столбец mg/mL означает API equivalents и x — обычную мольную долю, они задают implied solvent molarity c_solvent=(S_mass/MW)(1−x)/x. Сравним изменение этой величины, не подставляя выдуманную плотность.

In [4]:
group=[r for r in rows if r['solvent']=='methanol']
T=np.array([float(r['temperature_K']) for r in group]);x=np.array([float(r['x_as_reported']) for r in group])
xcalc=np.array([float(r['x_calculated_by_authors']) for r in group]);mass=np.array([float(r['solubility_mg_mL_as_reported']) for r in group])
derived_deviation=100*abs(xcalc-x)/x
implied_solvent_molarity=(mass/384.43)*(1-x)/x  # mg/mL numerically equals g/L.
display({'recomputed_relative_deviation_percent':derived_deviation.tolist(),
         'implied_solvent_molarity_mol_L':implied_solvent_molarity.tolist(),
         'relative_to_293K':(implied_solvent_molarity/implied_solvent_molarity[0]).tolist()})
z=1000/T;X=np.column_stack([np.ones(len(T)),z-z.mean()])
b_x=np.linalg.lstsq(X,np.log(x),rcond=None)[0]
b_mass=np.linalg.lstsq(X,np.log(mass),rcond=None)[0]
print('Apparent slopes, mole fraction and mass basis / kJ mol^-1:',-8.31446261815324*b_x[1],-8.31446261815324*b_mass[1])
fig,ax=plt.subplots();ax.plot(T,implied_solvent_molarity/implied_solvent_molarity[0],'o-')
ax.set(xlabel='T / K',ylabel='Implied solvent molarity relative to 293 K',title='Source-basis diagnostic, not measured solvent density')
publish_plot(fig,'solubility_basis_audit.png')
result={'kind':'calculated','scope':'primary solubility table reanalysis and source-consistency audit',
        'n_observations':len(rows),'conditional_fits':fits,'source_data_acceptance':'REVIEW_REQUIRED',
        'thermodynamic_enthalpy_accepted':False,'new_experiments':'NOT_RUN'}


{'recomputed_relative_deviation_percent': [0.8984725965858124,
  2.9579950582421257,
  1.1489536315141655,
  0.17859800565559902,
  0.0],
 'implied_solvent_molarity_mol_L': [27.299795017651203,
  17.481842069204568,
  11.331920474729138,
  7.9925857261503355,
  6.00037748756304],
 'relative_to_293K': [1.0,
  0.6403653235455192,
  0.41509177879915465,
  0.2927709061911482,
  0.21979569750188166]}

Apparent slopes, mole fraction and mass basis / kJ mol^-1: 66.2327912241552 35.222675409549055


<Figure size 640x480 with 1 Axes>

Метанольные столбцы дают ≈66.23 и ≈35.22 kJ/mol; implied solvent molarity к 333 K составляет лишь ≈0.220 от начальной. Требуется сверить dilution, единицы, состав раствора и определение x с raw data/авторами. Мы не исправляем таблицу по удобной гипотезе и не выбираем один столбец как «истину». Для солватов и смешанных растворителей отдельно проверяется термодинамическая основа. Численный OLS выполнен, физический параметр не принят.

In [5]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
print(append_record(init_dossier(),'M04-L02','A',result,ROOT,inputs=['practicuum/shared/data/istradefylline_solubility/istradefylline_solubility_table_S1.csv', 'practicuum/shared/data/istradefylline_solubility/istradefylline_aqueous_pH_table2.csv', 'practicuum/shared/data/istradefylline_solubility/supporting-information.pdf', 'practicuum/shared/data/istradefylline_solubility/primary-article.pdf', 'practicuum/shared/data/istradefylline_solubility/result.json', 'practicuum/shared/data/source-responses/statin_stability_verified.xml']))


/home/miakhlov/course-production-verified-20261118/practicuum/evidence/20261119-professional-solubility-linux/workspace/my_research/evidence/M04-L02/20261119T080231143376Z-A-e30cd393557845ae8b5eb266bd903ebf.json
